# 06 · Segmentation, and how a model plugs in later

Vibes Lite ships **classical segmentation** that needs no trained weights and
no GPU: thresholding, Otsu, multi-class banding and crop-health zones, refined
with OpenCV morphology and connected components.

The registry (`segmentation.register_model`) is the extension point: a
U-Net / DeepLab / YOLO-seg implementation registers itself under a name, and
the API, the dashboard and this notebook pick it up with no changes.

In [ ]:
%matplotlib inline
# --- Vibes Lite bootstrap -----------------------------------------------------
# These notebooks drive the processing engines directly (no HTTP layer needed).
import sys
import json
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = pathlib.Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "backend"))

import config, data, geospatial, preprocessing, indices, analysis
import opencv as vl_cv          # Vibes Lite's own engine (it wraps cv2 internally)
import segmentation, timeseries, visualization, storage

PATHS = config.ensure_directories()
NOTEBOOK_OUT = PATHS["outputs"] / "notebooks"
NOTEBOOK_OUT.mkdir(parents=True, exist_ok=True)

print("project root :", ROOT)
print("raw data     :", PATHS["raw_data"].relative_to(ROOT))
print("notebook out :", NOTEBOOK_OUT.relative_to(ROOT))

In [ ]:
DEMO_NAME = "synthetic_fields"


def demo_scene(name: str = DEMO_NAME, size: int = 900):
    """Return (raster, aoi, csv) for the labelled synthetic demo scene.

    The scene is only generated when missing. It is explicitly synthetic - its
    GeoTIFF tags carry SYNTHETIC=true - so nothing here pretends to be a real
    satellite acquisition.
    """
    raster = PATHS["raw_data"] / f"{name}.tif"
    aoi = PATHS["raw_data"] / f"{name}.geojson"
    table = PATHS["raw_data"] / f"{name}_ndvi_history.csv"
    if not raster.exists() or not aoi.exists():
        scene = data.generate_synthetic_scene(name=name, size=size, overwrite=True)
        return scene["paths"]["raster"], scene["paths"]["aoi"], scene["paths"]["table"]
    return raster, aoi, table


RASTER, AOI_FILE, HISTORY_CSV = demo_scene()
print("raster :", RASTER.name)
print("aoi    :", AOI_FILE.name)
print("history:", HISTORY_CSV.name)

## Prepare an index raster (the segmentation input)

In [ ]:
metadata = data.get_raster_metadata(RASTER)
resolution = indices.resolve_band_map(metadata, None)
aoi = data.load_aoi(AOI_FILE)

clipped = geospatial.clip_raster_to_aoi(
    str(RASTER), aoi["gdf"], bands=sorted(set(resolution["bands"].values())), max_pixels=config.max_analysis_pixels()
)
stack = preprocessing.band_stack(clipped, resolution["bands"])
valid = np.asarray(stack["valid_mask"])
ndvi = np.asarray(indices.compute_index("NDVI", stack["bands"], valid=valid)["array"], dtype="float32")
thresholds = config.health_thresholds()

print("classical methods :", segmentation.available_methods())
print("input raster      :", ndvi.shape, "| valid pixels:", int(valid.sum()))
print("health thresholds :", thresholds)

## 1. Threshold and Otsu: vegetation vs background

In [ ]:
vegetated = segmentation.threshold_segment(ndvi, threshold=0.4, comparison="above", valid=valid)
vegetated = np.asarray(vegetated)
print("vegetated pixels :", int(vegetated.sum()), "(%.1f%% of valid)" % (100 * vegetated.sum() / valid.sum()))

otsu = segmentation.otsu_segment(ndvi, invert=False, valid=valid)
print("otsu keys        :", sorted(otsu))
print("otsu threshold   :", otsu.get("threshold"))
print("otsu coverage    :", round(float(np.asarray(otsu["mask"]).mean()), 4))

## 2. Multi-class segmentation

The `multi_threshold` method bands the index using the configured crop-health breakpoints, so segmentation and crop health stay consistent.

In [ ]:
result = segmentation.run_segmentation(
    ndvi,
    method="multi_threshold",
    transform=stack["transform"],
    crs=stack["crs"],
    valid=valid,
    pixel_area_ha=stack["pixel_area_ha"],
    index_name="NDVI",
    min_area_ha=0.05,
)
print("method         :", result["method"], "| provider:", result["provider"])
print("mask coverage  :", result["mask_coverage_pct"], "% (", result["mask_pixels"], "px )")
print("components     :", result["component_count"], "| contours:", result["contour_count"])
print("morphology     :", result["morphology"])
print("class raster   :", None if result["class_raster"] is None else np.asarray(result["class_raster"]).shape)
print("area (ha)      :", result["area_ha"])

pd.DataFrame(result["statistics"]["classes"])

In [ ]:
class_raster = result["class_raster"]
picture = np.asarray(class_raster) if class_raster is not None else np.asarray(result["mask"], dtype="int16")

figure, axes = plt.subplots(1, 2, figsize=(12, 5.5))
axes[0].imshow(np.where(valid, ndvi, np.nan), cmap="RdYlGn", vmin=-0.2, vmax=1.0)
axes[0].set_title("input NDVI")
axes[0].axis("off")

axes[1].imshow(np.where(valid, picture, np.nan), cmap="YlGnBu")
axes[1].set_title("segmentation (" + result["method"] + ")")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 3. Objects: connected components and contours

Components give areas, centroids and bounding boxes - useful for trials, tree counts or per-plot reporting.

In [ ]:
components = segmentation.connected_components(np.asarray(result["mask"]), connectivity=8, min_area_pixels=64)
print("component count :", components["component_count"])
print("largest (px)    :", components["components"][0]["area_pixels"] if components["components"] else "n/a")

frame = pd.DataFrame(components["components"])
frame["area_ha"] = (frame["area_pixels"] * stack["pixel_area_ha"]).round(2)
frame[["label", "area_pixels", "area_ha", "centroid_pixels"]].head(6)

top_segments = segmentation.summarise_segments(result["segments"], top=5)
pd.DataFrame(top_segments)[["area_ha", "centroid_lonlat"]] if top_segments else print("No polygon segment above the minimum area.")

## 4. Export segments as GeoJSON

Segments carry the raster CRS, so they drop straight into QGIS, the Leaflet map or a later training-label review step.

In [ ]:
features = result["segments_geojson"]
print("features:", features["properties"]["segment_count"])
print("crs      :", features["features"][0]["geometry"]["type"] if features["features"] else "n/a")

export_path = NOTEBOOK_OUT / "06_segments.geojson"
export_path.write_text(json.dumps(features, indent=1))
print("written :", export_path.relative_to(ROOT), "(%.1f kB)" % (export_path.stat().st_size / 1024))

## 5. How a deep-learning model plugs in

The model registry keeps the platform AI-ready without shipping any weights. Registering a callable is the whole integration:

In [ ]:
print(json.dumps({name: {key: value for key, value in spec.items() if key != "loader"}
                   for name, spec in segmentation.MODEL_REGISTRY.items()}, indent=2)[:900])

In [ ]:
# Sketch only - the initial release ships no trained weights and needs no GPU.
def my_unet(array, **options):
    """A real implementation would load weights and return a mask and classes."""
    mask = np.asarray(array) > float(options.get("threshold", 0.5))
    classes = np.where(mask, 1, 0).astype("int16")
    return {"mask": mask, "classes": classes}


segmentation.register_model("unet-demo", loader=my_unet, task="segmentation", label="Notebook U-Net sketch")
print("available methods now:", segmentation.available_methods())

sketched = segmentation.run_segmentation(
    ndvi, method="unet-demo", params={"threshold": 0.5}, transform=stack["transform"], crs=stack["crs"],
    valid=valid, pixel_area_ha=stack["pixel_area_ha"], index_name="NDVI", min_area_ha=0.2,
)
print("provider:", sketched["provider"], "| components:", sketched["component_count"], "| coverage %:", sketched["mask_coverage_pct"])

**Why this matters.** The API endpoint `POST /api/segmentation` takes a
method name, so the day a real model is registered the dashboard can call it
with no backend or frontend change - and until then the classical methods keep
working, with no GPU and no download.